# HDO: alarma mensual con Youden exhaustivo

Dos celdas autocontenidas. Coloca el Excel revisado en esta carpeta y ejecútalas en orden. No se cargan módulos Python propios ni datos de Stage 2/3. W3/H4 sigue siendo la configuración principal.

La celda 1 estima cortes y métricas; la celda 2 representa ROC, ejemplos y atlas. Los IC son descriptivos, condicionados al corte seleccionado; no corrigen el optimismo. La selección de ejemplos no modifica la alarma.

In [ ]:
from pathlib import Path
from datetime import datetime
import re, json, hashlib, sys, scipy
import numpy as np
import pandas as pd

FILE = Path("1 178_audit_oct_2_2026_topics_revision(1).xlsx")
OUT = Path("HDO_alarmas_Youden_completo_" + datetime.now().strftime("%Y%m%d_%H%M%S"))
CONFIGS = [(3, 4), (3, 3), (4, 3), (4, 4)]
N_BOOT, SEED = 2000, 20261020
HISTORICAL_THRESHOLD = 1.610975

if not FILE.is_file():
    raise FileNotFoundError(f"No encuentro {FILE.name} en {Path.cwd()}. Corrige FILE.")
source_hash = hashlib.sha256(FILE.read_bytes()).hexdigest()
df = pd.read_excel(FILE)
df.columns = df.columns.astype(str).str.strip()
required = ["id", "Evento_DB1", "Tiempo_obs_DB1", "topic_all",
            "Fecha_inicio_actualizada", "Fecha_Fin_DB1_Actualizado"]
if any(c not in df.columns for c in required):
    raise ValueError("Faltan columnas: " + str([c for c in required if c not in df]))
df = df.loc[df["id"].notna()].copy().reset_index(drop=True)
for c in required[:4]:
    df[c] = pd.to_numeric(df[c], errors="raise")
if (df["id"] % 1 != 0).any() or df["id"].duplicated().any():
    raise ValueError("IDs no enteros o duplicados.")
if not df["Evento_DB1"].isin([0, 1]).all():
    raise ValueError("Evento_DB1 debe ser 0/1, sin ausentes.")
if not np.isfinite(df["Tiempo_obs_DB1"]).all() or (df["Tiempo_obs_DB1"] < 0).any():
    raise ValueError("Duraciones no válidas.")
df["id"] = df["id"].astype(int)
df["Evento_DB1"] = df["Evento_DB1"].astype(int)
start = pd.to_datetime(df["Fecha_inicio_actualizada"], errors="raise")
end = pd.to_datetime(df["Fecha_Fin_DB1_Actualizado"], errors="raise")
if not np.allclose((end-start).dt.total_seconds()/86400, df["Tiempo_obs_DB1"], atol=1e-7, rtol=0):
    raise ValueError("La duración no coincide con fin menos inicio.")
months = sorted({int(m.group(1)) for c in df.columns
                 if (m := re.fullmatch(r"mes(\d+)_topic[0-5]", c))})
if not months or months != list(range(1, max(months)+1)):
    raise ValueError("Los meses deben ser consecutivos desde mes1.")
cols = [[f"mes{m}_topic{k}" for k in range(6)] for m in months]
if any(c not in df for group in cols for c in group):
    raise ValueError("Cada mes debe tener seis columnas de probabilidades.")
P = np.stack([df[c].apply(pd.to_numeric, errors="raise").to_numpy(float) for c in cols], axis=1)
n_present = np.sum(~np.isnan(P), axis=2)
if not np.isin(n_present, [0, 6]).all():
    raise ValueError("Hay posteriores incompletos: no se rellenan con ceros.")
valid = n_present == 6
if (not np.isfinite(P[valid]).all() or (P[valid] < 0).any()
        or (P[valid] > 1).any() or (np.abs(P[valid].sum(axis=1)-1) > 1e-6).any()):
    raise ValueError("Probabilidades no válidas; no se normalizan automáticamente.")
pUF = P[:, :, 3:6].sum(axis=2)
OUT.mkdir(parents=True, exist_ok=False)

def metrics(d, theta, cluster_weights=None):
    y, x = d["y"].to_numpy(int), d["score"].to_numpy(float)
    w = np.ones(len(d)) if cluster_weights is None else cluster_weights
    pred = x >= theta
    pos, neg = w*y, w*(1-y)
    tp, fn, fp, tn = pos[pred].sum(), pos[~pred].sum(), neg[pred].sum(), neg[~pred].sum()
    ratio = lambda a, b: float(a/b) if b else np.nan
    _, ix = np.unique(x, return_inverse=True)
    pg, ng = np.bincount(ix, weights=pos), np.bincount(ix, weights=neg)
    auc = ratio(np.sum(pg*(np.cumsum(ng)-ng/2)), pos.sum()*neg.sum())
    return dict(AUROC=auc, sensitivity=ratio(tp,tp+fn), specificity=ratio(tn,tn+fp),
                PPV=ratio(tp,tp+fp), NPV=ratio(tn,tn+fn), TP=tp, FN=fn, FP=fp, TN=tn)

def complete_sweep(d):
    ordered = d.sort_values("score", ascending=False, kind="stable")
    x, y = ordered["score"].to_numpy(), ordered["y"].to_numpy(int)
    npos, nneg = int(y.sum()), int((1-y).sum())
    if not npos or not nneg:
        raise ValueError("Se necesitan puntos positivos y negativos.")
    last = np.r_[np.flatnonzero(np.diff(x)), len(x)-1]
    tp = np.r_[0, np.cumsum(y)[last]]
    fp = np.r_[0, last+1-np.cumsum(y)[last]]
    sw = pd.DataFrame(dict(threshold=np.r_[np.inf,x[last]], TP=tp, FP=fp,
                           FN=npos-tp, TN=nneg-fp))
    sw["sensitivity"], sw["specificity"] = tp/npos, 1-fp/nneg
    sw["J_numerator"] = tp*nneg-fp*npos  # Comparación exacta de Youden.
    sw["J"] = sw["J_numerator"]/(npos*nneg)
    tied = sw.loc[sw["J_numerator"] == sw["J_numerator"].max()]
    # Regla explícita para empates: menor corte. Los cuatro máximos actuales son únicos.
    theta = float(tied["threshold"].min())
    sw["selected"] = sw["threshold"] == theta
    return theta, sw, len(tied)

SETS, ALL_POINTS, SWEEPS, SUMMARY = {}, {}, {}, []
for W, H in CONFIGS:
    rows = []
    for i, row in df.iterrows():
        duration, event = float(row["Tiempo_obs_DB1"]), int(row["Evento_DB1"])
        for m in months:
            T = 30*m
            if m < W or T >= duration:
                continue
            window = pUF[i, m-W:m]
            if not np.isfinite(window).all():
                continue
            y = 1 if event and duration <= T+30*H else (0 if duration >= T+30*H else np.nan)
            rows.append(dict(id=int(row["id"]), month=m, day=T,
                             decision_date=(start.iloc[i]+pd.Timedelta(days=T)).date().isoformat(),
                             horizon_end=T+30*H, duration=duration, event=event,
                             pUF=float(pUF[i,m-1]), score=float(window.sum()), y=y))
    points = pd.DataFrame(rows)
    d = points.dropna(subset=["y"]).copy().sort_values(["id","month"]).reset_index(drop=True)
    d["y"] = d["y"].astype(int)
    theta, sw, n_tied = complete_sweep(d)
    for frame in (points,d):
        frame["threshold"] = theta
        frame["alarm"] = frame["score"] >= theta
    result = dict(W=W,H=H,N_patients=d["id"].nunique(),N_points=len(d),
                  positive_points=int(d["y"].sum()), negative_points=int((1-d["y"]).sum()),
                  unique_events=d.loc[d["y"]==1,"id"].nunique(),
                  threshold=theta, threshold_mean=theta/W, n_tied_maxima=n_tied,
                  **metrics(d,theta))
    result["J"] = result["sensitivity"]+result["specificity"]-1
    # IC descriptivos: bootstrap por paciente, SIN volver a seleccionar el corte.
    ids, ix = np.unique(d["id"], return_inverse=True)
    rng, boots = np.random.default_rng(SEED), []
    names = ["AUROC","sensitivity","specificity","PPV","NPV"]
    for _ in range(N_BOOT):
        counts = np.bincount(rng.integers(0,len(ids),len(ids)), minlength=len(ids))
        boots.append(metrics(d,theta,counts[ix]))
    for name in names:
        values = np.array([b[name] for b in boots])
        values = values[np.isfinite(values)]
        result[name+"_CI_low"], result[name+"_CI_high"] = (np.percentile(values,[2.5,97.5]) if len(values) else [np.nan,np.nan])
        result[name+"_bootstrap_valid"] = len(values)
    SETS[W,H], ALL_POINTS[W,H], SWEEPS[W,H] = d, points, sw
    SUMMARY.append(result)
    for label, frame in [("puntos_evaluables",d),("puntos_con_score",points),("barrido",sw)]:
        frame.to_csv(OUT/f"{label}_W{W}_H{H}.csv",index=False,float_format="%.17g")

SUMMARY = pd.DataFrame(SUMMARY)
SUMMARY.to_csv(OUT/"resumen_metricas.csv",index=False,float_format="%.17g")
primary = SUMMARY.loc[(SUMMARY.W==3)&(SUMMARY.H==4)].iloc[0]
THETA = float(primary.threshold)
D = ALL_POINTS[3,4]
comparison = pd.DataFrame([dict(rule=label,threshold=t,**metrics(SETS[3,4],t))
    for label,t in [("historical",HISTORICAL_THRESHOLD),("reestimated_exhaustive",THETA)]])
comparison.to_csv(OUT/"comparacion_historico_nuevo.csv",index=False,float_format="%.17g")
if hashlib.sha256(FILE.read_bytes()).hexdigest() != source_hash:
    raise RuntimeError("El archivo de entrada ha cambiado durante el análisis.")
(OUT/"manifest.json").write_text(json.dumps(dict(input=FILE.name,sha256=source_hash,
    primary=[3,4],threshold=THETA,selection="Exhaustive Youden; lowest cut if tied; score >= cut",
    bootstrap=N_BOOT,seed=SEED,CI_scope="Fixed selected cut; no optimism correction",
    scope="Apparent development performance; DB1 only; no representation refit",
    source_N=len(df),source_events=int(df.Evento_DB1.sum()),
    code_version="HDO-DB1-exhaustive-2cells-v1.0-20261022",
    versions=dict(python=sys.version.split()[0],numpy=np.__version__,pandas=pd.__version__,scipy=scipy.__version__)),indent=2),encoding="utf-8")
print(SUMMARY[["W","H","N_patients","N_points","threshold","AUROC",
               "sensitivity","specificity","PPV","NPV"]].to_string(index=False))
print(f"\nCorte W3/H4 = {THETA:.17g}; media = {THETA/3:.17g}\nSalida: {OUT.resolve()}")


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

# Esta celda utiliza los resultados de la celda 1; no reestima el corte.
MAIN_IDS = [91008, 23003, 41020, 71002]
SUPPLEMENT_IDS = [11001, 23004]
ALTERNATIVE_IDS = [61002, 62004]
SHOW_EXAMPLES = True
BUILD_ATLAS = True
plt.rcParams.update({"pdf.fonttype":42,"ps.fonttype":42,"svg.fonttype":"none"})

def save_plot(fig, name):
    for ext in ("png","pdf","svg"):
        fig.savefig(OUT/f"{name}.{ext}",dpi=300,bbox_inches="tight")

fig, ax = plt.subplots(figsize=(7.4,5.6))
for W,H in CONFIGS:
    sw = SWEEPS[W,H]
    r = SUMMARY.loc[(SUMMARY.W==W)&(SUMMARY.H==H)].iloc[0]
    line, = ax.plot(1-sw.specificity,sw.sensitivity,
        label=f"W={W}, H={H}: AUROC {r.AUROC:.3f}; cut {r.threshold:.4f}")
    ax.plot(1-r.specificity,r.sensitivity,marker="o",linestyle="none",color=line.get_color())
ax.plot([0,1],[0,1],linestyle="--",linewidth=0.8,label="Chance")
ax.set(xlim=(0,1),ylim=(0,1.02),xlabel="False positive rate (1 - specificity)",
       ylabel="True positive rate (sensitivity)",title="Monthly alarm: audited Stage 1")
ax.legend(loc="lower right",fontsize=8,frameon=False)
fig.tight_layout()
save_plot(fig,"Fig5C_ROC_Youden_completo")
plt.show()
plt.close(fig)

id_to_row = {int(pid):i for i,pid in enumerate(df["id"])}

def patient_plot(pid):
    if pid not in id_to_row:
        raise ValueError(f"ID ausente: {pid}")
    i = id_to_row[pid]
    row = df.iloc[i]
    duration, event = float(row["Tiempo_obs_DB1"]), int(row["Evento_DB1"])
    g = D.loc[D.id==pid].sort_values("month")
    e = g.dropna(subset=["y"])
    alarms = g.loc[g.alarm]
    tp = e.loc[(e.y==1)&e.alarm]
    counts = dict(TP=len(tp), FN=int(((e.y==1)&~e.alarm).sum()),
                  FP=int(((e.y==0)&e.alarm).sum()), TN=int(((e.y==0)&~e.alarm).sum()))
    fig, ax = plt.subplots(figsize=(9.4,5.1))
    t = 30*np.array(months,dtype=float)
    # No dibujar datos de bloques que comienzan después del intervalo autorizado.
    visible = (t-30 < duration)
    values = pUF[i].copy()
    values[~visible] = np.nan
    t = np.minimum(t,duration)  # El bloque terminal parcial se sitúa en el fin real.
    ax.plot(t[visible],values[visible],marker=".",markersize=4,linewidth=1,label="Monthly pUF")
    # Reindexar conserva los huecos: nunca unirlos como meses consecutivos.
    rolling = g.set_index("month")["score"].reindex(months)/3
    ax.plot(30*np.array(months),rolling,marker=".",markersize=4,linewidth=1.3,label="Rolling burden / 3")
    ax.axhline(THETA/3,linestyle="--",linewidth=0.9,label=f"Cut: {THETA/3:.6f}")
    first = float(alarms.day.min()) if len(alarms) else None
    first_tp = float(tp.day.min()) if len(tp) else None
    if first is not None:
        ax.axvline(first,linestyle=":",linewidth=1,label=f"First alarm: day {first:g}")
    if first_tp is not None and first_tp != first:
        ax.axvline(first_tp,linestyle="--",linewidth=1,label=f"First TP: day {first_tp:g}")
    endpoint = "PFS event" if event else "Censoring"
    ax.axvline(duration,linestyle="-.",linewidth=1,label=f"{endpoint}: day {duration:g}")
    unresolved = g.loc[g.y.isna()]
    if len(unresolved):
        ax.plot(unresolved.day,unresolved.score/3,linestyle="none",marker="o",
                markerfacecolor="none",markersize=6,label="Unresolved horizon")
    if not np.isfinite(values).any():
        ax.text(.5,.5,"No monthly posteriors available",transform=ax.transAxes,ha="center")
    elif g.empty:
        ax.text(.5,.5,"No evaluable rolling history before endpoint",transform=ax.transAxes,ha="center")
    ax.set(xlim=(0,max(120,duration)*1.04),ylim=(-.02,1.05),xlabel="Days from treatment start",
           ylabel="pUF / normalized burden",title=f"Participant {pid} | {row.get('Cohorte','')}")
    ax.legend(loc="upper left",bbox_to_anchor=(1.01,1),fontsize=7.5,frameon=False)
    text = " | ".join(f"{k}: {v}" for k,v in counts.items())
    text += f" | Unresolved horizons: {len(unresolved)}"
    if first_tp is not None:
        text += f" | Lead from first TP: {duration-first_tp:g} days"
    fig.text(.1,.055,text,fontsize=8)
    fig.text(.1,.025,"Monthly points; terminal partial block is descriptive only. TP/FP/FN/TN count decision points, not episodes.",fontsize=7)
    fig.subplots_adjust(left=.1,right=.72,bottom=.18,top=.89)
    record = dict(id=pid,duration_days=duration,event=event,**counts,
                  first_alarm_day=first,first_TP_day=first_tp,
                  lead_from_first_TP=duration-first_tp if first_tp is not None else None,
                  unresolved_horizons=len(unresolved))
    return fig,record

selected = MAIN_IDS+SUPPLEMENT_IDS+ALTERNATIVE_IDS
selected_records = []
with PdfPages(OUT/"Ejemplos_propuestos.pdf") as book:
    for pid in selected:
        fig, rec = patient_plot(pid)
        book.savefig(fig)
        save_plot(fig,f"Paciente_{pid}")
        selected_records.append(rec)
        if SHOW_EXAMPLES and pid in MAIN_IDS+SUPPLEMENT_IDS:
            plt.show()
        plt.close(fig)
pd.DataFrame(selected_records).to_csv(OUT/"ejemplos_propuestos.csv",index=False)

if BUILD_ATLAS:
    roster = df.loc[(df.Tiempo_obs_DB1>=28)&df.topic_all.notna(),"id"].sort_values()
    index = []
    with PdfPages(OUT/"Atlas_Youden_completo.pdf") as book:
        for page,pid in enumerate(roster,1):
            fig, rec = patient_plot(int(pid))
            fig.text(.93,.025,str(page),ha="right",fontsize=7)
            book.savefig(fig)
            plt.close(fig)
            index.append(dict(page=page,**rec))
    pd.DataFrame(index).to_csv(OUT/"indice_atlas_interno.csv",index=False)
    print(f"Atlas: {len(index)} páginas. Esta población es de visualización, no el denominador de la ROC.")
print(f"Figuras, tablas y PDF guardados en: {OUT.resolve()}")
